# KB AI Challenge 전체 파이프라인 실행

## 1. 실행 환경 호환성 설정

In [1]:
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path
import os
import subprocess
import sys

REQUIRED_SKLEARN_VERSION = "1.5.2"

try:
    current_sklearn_version = version("scikit-learn")
except PackageNotFoundError:
    current_sklearn_version = None

print("Python 실행 파일:", sys.executable)
print("현재 scikit-learn:", current_sklearn_version)

if current_sklearn_version != REQUIRED_SKLEARN_VERSION:
    print(f"scikit-learn을 {REQUIRED_SKLEARN_VERSION}로 맞춥니다.")
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--no-cache-dir",
            "--force-reinstall",
            "--no-deps",
            f"scikit-learn=={REQUIRED_SKLEARN_VERSION}",
        ],
        check=True,
    )
    print("설치된 scikit-learn:", version("scikit-learn"))
else:
    print("scikit-learn 버전이 이미 맞습니다.")


Python 실행 파일: <python-env>\python.exe
현재 scikit-learn: 1.5.2
scikit-learn 버전이 이미 맞습니다.


## 2. 실행 설정

In [6]:
from datetime import datetime
from pathlib import Path

CODE_DIR = Path.cwd().resolve()
DATA_DIR = CODE_DIR / "data"
EXECUTED_ROOT_DIR = CODE_DIR / "executed_notebooks"

DATA_DIR.mkdir(parents=True, exist_ok=True)
EXECUTED_ROOT_DIR.mkdir(parents=True, exist_ok=True)

RUN_MODE = "model_action"
TIMEOUT_SECONDS = 14400
KERNEL_NAME = "python3"

FULL_NOTEBOOK_ORDER = [
    "step1_preprocessing.ipynb",
    "step1_model.ipynb",
    "merged_preprocessing.ipynb",
    "step2_preprocessing.ipynb",
    "step2_model.ipynb",
    "action_plan.ipynb",
]

MODEL_ACTION_NOTEBOOK_ORDER = [
    #"step1_model.ipynb",
    #"step2_model.ipynb",
    "action_plan.ipynb",
]

STEP2_DEBUG_NOTEBOOK_ORDER = [
    "step2_model.ipynb",
]

NOTEBOOK_ORDERS = {
    "full": FULL_NOTEBOOK_ORDER,
    "model_action": MODEL_ACTION_NOTEBOOK_ORDER,
    "step2_debug": STEP2_DEBUG_NOTEBOOK_ORDER,
}

if RUN_MODE not in NOTEBOOK_ORDERS:
    raise ValueError(
        f"지원하지 않는 RUN_MODE입니다: {RUN_MODE}. "
        f"가능한 값: {list(NOTEBOOK_ORDERS)}"
    )

NOTEBOOK_ORDER = NOTEBOOK_ORDERS[RUN_MODE]
RUN_STARTED_AT = datetime.now()
RUN_STAMP = RUN_STARTED_AT.strftime("%Y%m%d_%H%M%S")
EXECUTED_DIR = EXECUTED_ROOT_DIR / f"{RUN_MODE}_{RUN_STAMP}"
EXECUTED_DIR.mkdir(parents=True, exist_ok=True)

print("코드 폴더:", CODE_DIR)
print("데이터 폴더:", DATA_DIR)
print("실행 모드:", RUN_MODE)
print("실행 순서:", NOTEBOOK_ORDER)
print("실행본 저장 폴더:", EXECUTED_DIR)


코드 폴더: .
데이터 폴더: .\data
실행 모드: model_action
실행 순서: ['action_plan.ipynb']
실행본 저장 폴더: .\executed_notebooks\model_action_20260803_150217


## 3. 노트북 실행 함수

In [8]:
import subprocess
import sys
from pathlib import Path


def run_notebook(
    notebook_path: Path,
    executed_dir: Path,
    timeout_seconds: int = 14400,
    kernel_name: str = "python3",
) -> Path:
    notebook_path = notebook_path.resolve()
    executed_dir = executed_dir.resolve()
    executed_dir.mkdir(parents=True, exist_ok=True)

    if not notebook_path.exists():
        raise FileNotFoundError(f"실행할 노트북이 없습니다: {notebook_path}")

    output_path = executed_dir / f"{notebook_path.stem}_executed.ipynb"
    log_path = executed_dir / f"{notebook_path.stem}_execution.log"

    command = [
        sys.executable,
        "-m",
        "jupyter",
        "nbconvert",
        "--to",
        "notebook",
        "--execute",
        str(notebook_path),
        "--output",
        output_path.name,
        "--output-dir",
        str(executed_dir),
        f"--ExecutePreprocessor.timeout={timeout_seconds}",
        f"--ExecutePreprocessor.kernel_name={kernel_name}",
        "--ExecutePreprocessor.allow_errors=False",
    ]

    child_env = os.environ.copy()
    child_env["PYTHONUTF8"] = "1"
    child_env["PYDEVD_DISABLE_FILE_VALIDATION"] = "1"

    print("\n" + "=" * 80)
    print("실행 시작:", notebook_path.name)
    print("=" * 80)

    result = subprocess.run(
        command,
        cwd=str(CODE_DIR),
        env=child_env,
        text=True,
        capture_output=True,
        encoding="utf-8",
        errors="replace",
    )

    full_log = (
        "===== COMMAND =====\n"
        + " ".join(command)
        + "\n\n===== STDOUT =====\n"
        + result.stdout
        + "\n\n===== STDERR =====\n"
        + result.stderr
    )
    log_path.write_text(full_log, encoding="utf-8")

    if result.returncode != 0:
        error_tail = result.stderr[-20000:] if result.stderr else result.stdout[-20000:]
        print("\n❌ 노트북 실행 실패")
        print(error_tail)
        print("\n전체 로그:", log_path)
        raise RuntimeError(
            f"노트북 실행 실패: {notebook_path.name}\n"
            f"전체 로그: {log_path}"
        )

    if not output_path.exists():
        raise RuntimeError(
            f"명령은 종료됐지만 실행본이 생성되지 않았습니다: {output_path}"
        )

    print("✅ 실행 완료:", notebook_path.name)
    print("실행본:", output_path)
    print("로그:", log_path)
    return output_path


## 4. 실행 전 점검

In [9]:
missing_notebooks = [
    notebook_name
    for notebook_name in NOTEBOOK_ORDER
    if not (CODE_DIR / notebook_name).exists()
]

if missing_notebooks:
    raise FileNotFoundError(
        "다음 노트북을 찾을 수 없습니다:\n- "
        + "\n- ".join(missing_notebooks)
    )

print("실행 대상 노트북 확인 완료")
for notebook_name in NOTEBOOK_ORDER:
    print(" -", CODE_DIR / notebook_name)


실행 대상 노트북 확인 완료
 - .\action_plan.ipynb


## 5. 선택한 파이프라인 실행

In [10]:
completed_notebooks = []
executed_notebook_paths = []
failed_notebook = None

try:
    for notebook_name in NOTEBOOK_ORDER:
        notebook_path = CODE_DIR / notebook_name

        executed_path = run_notebook(
            notebook_path=notebook_path,
            executed_dir=EXECUTED_DIR,
            timeout_seconds=TIMEOUT_SECONDS,
            kernel_name=KERNEL_NAME,
        )

        completed_notebooks.append(notebook_name)
        executed_notebook_paths.append(executed_path)

except Exception:
    failed_notebook = notebook_name
    print("\n중단된 노트북:", failed_notebook)
    print("완료된 노트북:", completed_notebooks)
    print("실행 로그 폴더:", EXECUTED_DIR)
    raise

print("\n" + "=" * 80)
print("전체 실행 완료")
print("=" * 80)
print("완료된 노트북:", completed_notebooks)
print("실행본 저장 폴더:", EXECUTED_DIR)



실행 시작: action_plan.ipynb
✅ 실행 완료: action_plan.ipynb
실행본: .\executed_notebooks\model_action_20260803_150217\action_plan_executed.ipynb
로그: .\executed_notebooks\model_action_20260803_150217\action_plan_execution.log

전체 실행 완료
완료된 노트북: ['action_plan.ipynb']
실행본 저장 폴더: .\executed_notebooks\model_action_20260803_150217


## 6. 이번 실행에서 생성되거나 수정된 파일 확인

In [11]:
from datetime import datetime
import pandas as pd

records = []
start_timestamp = RUN_STARTED_AT.timestamp()

for path in CODE_DIR.rglob("*"):
    if not path.is_file():
        continue

    stat = path.stat()
    if stat.st_ctime < start_timestamp and stat.st_mtime < start_timestamp:
        continue

    records.append(
        {
            "파일명": path.name,
            "상대경로": str(path.relative_to(CODE_DIR)),
            "생성시각": datetime.fromtimestamp(stat.st_ctime).strftime(
                "%Y-%m-%d %H:%M:%S"
            ),
            "수정시각": datetime.fromtimestamp(stat.st_mtime).strftime(
                "%Y-%m-%d %H:%M:%S"
            ),
            "크기_MB": round(stat.st_size / (1024 ** 2), 2),
        }
    )

created_or_modified_df = pd.DataFrame(records)

if created_or_modified_df.empty:
    print("이번 실행에서 생성되거나 수정된 파일이 없습니다.")
else:
    created_or_modified_df = created_or_modified_df.sort_values(
        ["수정시각", "생성시각"],
        ascending=False,
    ).reset_index(drop=True)
    display(created_or_modified_df)
    print(f"총 {len(created_or_modified_df):,}개 파일")


,파일명,상대경로,생성시각,수정시각,크기_MB
0,action_plan_executed.ipynb,executed_notebooks\model_action_20260803_15021...,2026-08-03 15:10:36,2026-08-03 15:10:36,0.16
1,action_plan_execution.log,executed_notebooks\model_action_20260803_15021...,2026-08-03 15:10:36,2026-08-03 15:10:36,0.00
2,action_plan_representative_cases.csv,result\action_plan\action_plan_representative_...,2026-08-03 13:13:09,2026-08-03 15:10:34,0.01
3,action_plan_result.csv,result\action_plan\action_plan_result.csv,2026-08-03 13:12:58,2026-08-03 15:10:34,100.35
4,feature_cols.pkl,result\step2_model\saved_models_v6\feature_col...,2026-08-03 15:03:05,2026-08-03 15:03:05,0.00
5,final_calibrator_for_service.pkl,result\step2_model\saved_models_v6\final_calib...,2026-08-03 15:03:05,2026-08-03 15:03:05,1.06
6,final_model_for_shap.pkl,result\step2_model\saved_models_v6\final_model...,2026-08-03 15:03:05,2026-08-03 15:03:05,1.06
7,fixed_params.pkl,result\step2_model\saved_models_v6\fixed_param...,2026-08-03 15:03:05,2026-08-03 15:03:05,0.00


총 8개 파일
